Code to format the btk output into deepdisc friendly images+annotations.  Not finalized!

This notebook will

0. Convert the CosmoDC2 catalog to the 
1. Produce a btk blend image from a CatSim catalog
2. Create the annotations for the image.  
3. Check by using deepdisc visualization to look at objects in the image



In [ ]:
# Import the neccessary packages

import btk
import btk.survey
import btk.draw_blends
import btk.catalog
import btk.sampling_functions
import cv2
import numpy as np
from galcheat.utilities import mean_sky_level
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table, vstack


### 0) Convert the CosmoDC2 catalog to the liking of the BTK, and save

In [ ]:
# fits_catalog_name = "data/cosmoDC2_6bands_1deg.fits"
# fits_catalog = fits.open(fits_catalog_name)[1].data

In [ ]:

# def e1e2_to_ephi(e1,e2):
    
#     pa = np.arctan(e2/e1)
    
#     return pa

# def fits_catalog_to_btk_catalog_all_bands(catalog):
    
#     L0 = 3.0128e28
    
#     btk_catalog = Table([catalog['ra']], names = ['ra'])
#     btk_catalog['dec'] = catalog['dec']
#     btk_catalog['redshift'] = catalog['redshift']
    
#     btk_catalog['a_b'] = catalog['size_bulge_true']
#     btk_catalog['b_b'] = catalog['size_minor_bulge_true']
    
#     btk_catalog['a_d'] = catalog['size_disk_true']
#     btk_catalog['b_d'] = catalog['size_minor_disk_true']
    
    
#     btk_catalog['pa_bulge'] = e1e2_to_ephi(catalog['ellipticity_1_bulge_true'],catalog['ellipticity_2_bulge_true']) * 180.0/np.pi
    
#     btk_catalog['pa_disk'] = e1e2_to_ephi(catalog['ellipticity_1_disk_true'],catalog['ellipticity_2_disk_true']) * 180.0/np.pi
    
#     for band in ['u', 'g', 'r', 'i', 'z', 'y']:
        
#         total_flux = L0 * 10**(-0.4*catalog[f'mag_true_{band}'])
#         bulge_to_total_ratio = catalog[f'bulge_to_total_ratio_{band}']

#         btk_catalog[f'fluxnorm_bulge_{band}'] = total_flux * bulge_to_total_ratio
#         btk_catalog[f'fluxnorm_disk_{band}'] = total_flux * (1-bulge_to_total_ratio)
#         btk_catalog[f'fluxnorm_agn_{band}'] = np.zeros(total_flux.shape)
        
#         btk_catalog[f'{band}_ab'] = catalog[f'mag_true_{band}']
    
#     return btk_catalog
    

In [ ]:

# catalog = fits_catalog_to_btk_catalog_all_bands(fits_catalog)

# catalog.write(f'data/btk_catalog_1deg_6bands.fits', format = 'fits', overwrite = True)




#

### 1) Produce a blended image from a catalog.  A lot of this follows the BTK quickstart tutorial

In [ ]:
catalog_name = "data/btk_catalog_1deg_6bands.fits" # contains ~272k entries

catalog = btk.catalog.CatsimCatalog.from_file(catalog_name)
catalog.table[:5] # display 5 first entries of table containing the actual catalog information.

In [ ]:
#stamp_size = 24.0  # Size of the stamp, in arcseconds
stamp_size = 24.0  # Size of the stamp, in arcseconds

max_number = 100    # Maximum number of galaxies in a blend
seed = 0 # random seed for reproducibility purposes # seed = 8
sampling_function = btk.sampling_functions.RandomSquareSampling(
    max_number=max_number,  # always get `max_number` galaxies
    stamp_size=stamp_size,
    min_mag = 20.0, max_mag = 25.3,
    seed = seed)

In [ ]:
blend_catalog = sampling_function(catalog.table)
blend_catalog # ra and dec are now relative to the center of the blend (not the original ones)

In [ ]:
import galsim

In [ ]:
LSST = btk.survey.get_surveys("LSST")
# for band in ['u', 'g', 'r', 'i', 'z', 'y']:
#     LSST.get_filter(band).psf = galsim.Gaussian(sigma = 0.01)

In [ ]:
batch_size = 1

draw_generator = btk.draw_blends.CatsimGenerator(
    catalog,
    sampling_function,
    LSST,
    batch_size=batch_size,
    stamp_size=stamp_size,
    njobs=1,
    add_noise="all",
    seed=seed, # use same seed here
)

# generate batch 100 blend catalogs and images.
blend_batch = next(draw_generator)

In [ ]:
blend_batch

In [ ]:
from btk.plotting import get_rgb

im =  blend_batch.blend_images[0]
bands = [2, 3, 4] # r, i, z
rgb = get_rgb(im[bands])


plt.imshow(rgb)

### 2) Format the images to deepdisc metadata

This will create footprints and bounding boxes from the i-band isolated truth images

In [ ]:
def get_bbox(mask):
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    return rmin-1, rmax+1, cmin-1, cmax+1

In [ ]:
#from detectron2.structures import BoxMode

# We use the i-band image for segmentation and footprint by interpreting the unblended image from blend batch
sky_level = mean_sky_level(LSST, 'i').to_value('electron') # gain = 1
idx = 0

def create_metadata(blend_batch, sky_level, sn, idx):

    """ Code to format the metadatain to a dict.  It takes the i-band and makes a footprint+bounding boxes
    from thresholding to sn*sky_level
    
    Parameters
    
    blend_batch: BTK blend batch
        BTK batch of blends
    sky_level: float
        The background sky level in the i-band
    sn: int
        The signal-to-noise ratio for thresholding
    idx:
        The index of the blend in the blend_batch
        
    Returns
        ddict: dict
            The dictionary of metadata for the idx'th blend in the batch 
    
    """
    

    ddict = {}

    ddict[f"file_name"] = 'none'
    ddict["image_id"] = idx
    ddict["height"] = blend_batch.isolated_images.shape[-2]
    ddict["width"] = blend_batch.isolated_images.shape[-1]

    iso_images = blend_batch.isolated_images[:,:, 3] # only i-band for now
    segs = btk.metrics.utils.get_segmentation(iso_images, sky_level, sigma_noise=sn)
#     print(segs.shape)
#     plt.imshow(segs[0][1])


    cat = blend_batch.catalog_list[idx]
    n = len(cat)
    objs = []
    for j in range(n):
        detect = iso_images[idx][j]
        mask = segs[idx][j]

        bbox = get_bbox(mask)
        mask = segs[idx][j]
        x0 = bbox[2]
        x1 = bbox[3]
        y0 = bbox[0]
        y1 = bbox[1]


        w = x1-x0
        h = y1-y0
        bbox = [x0, y0, w, h]

        redshift = cat[j]['redshift']

        contours, hierarchy = cv2.findContours(
                    (mask).astype(np.uint8), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
                )


        segmentation = []
        for contour in contours:
            # contour = [x1, y1, ..., xn, yn]
            contour = contour.flatten()
            if len(contour) > 4:
                segmentation.append(contour.tolist())
        # No valid countors
        if len(segmentation) == 0:
            continue

        obj = {
            "bbox": bbox,
            "area": w*h,
            #"bbox_mode": BoxMode.XYWH_ABS,
            "bbox_mode": 1,
            "segmentation": segmentation,
            "category_id": 0,
            "redshift": redshift
        }
        objs.append(obj)

    ddict['annotations'] = objs

    return ddict

In [ ]:
ddict = create_metadata(blend_batch,sky_level,sn=2,idx=0)

### 3) This will visualize the annotations using detectron2/deepdisc functions

In [ ]:
# To do this, we need detectron2 installed, as well as lincc-framework/deepdisc

from detectron2.data import MetadataCatalog, DatasetCatalog

if "astro_test" in DatasetCatalog.list():
    print('removing astro_test')
    DatasetCatalog.remove("astro_test")
    MetadataCatalog.remove("astro_test")

In [ ]:
from matplotlib import colors

if "astro_test" in DatasetCatalog.list():
    print('removing astro_test')
    DatasetCatalog.remove("astro_test")
    MetadataCatalog.remove("astro_test")

    
red = np.array(colors.to_rgb('red'))*255
white = np.array(colors.to_rgb('white'))*255
blue = np.array(colors.to_rgb('blue'))*255
green = np.array(colors.to_rgb('green'))*255

DatasetCatalog.register("astro_test", lambda: np.load(testfile))

astrotest_metadata = MetadataCatalog.get("astro_test").set(thing_classes=["object"]).set(thing_colors=[green])
#astrotest_metadata = register_data_set("astro_test", testfile, np.load, thing_classes=classes, thing_colors=green)

In [ ]:
import matplotlib.pyplot as plt
from deepdisc.astrodet.visualizer import ColorMode
from deepdisc.astrodet.visualizer import Visualizer
from astropy.visualization import make_lupton_rgb

image = blend_batch.blend_images[idx]
b1 = image[2]
b2 = image[1]
b3 = image[0]
stretch = np.max(image) - np.min(image)
Q = 0.1
img = make_lupton_rgb(b1, b2, b3, minimum=0, stretch=stretch, Q=Q)

print("total instances:", len(ddict["annotations"]))
v0 = Visualizer(
    img,
    metadata=astrotest_metadata,
    scale=1,
    instance_mode=ColorMode.SEGMENTATION,  # remove the colors of unsegmented pixels. This option is only available for segmentation models
)
groundTruth = v0.draw_dataset_dict(ddict, lf=False, alpha=0.1, boxf=True)

ax1 = plt.subplot(1, 1, 1)
ax1.imshow(groundTruth.get_image())
ax1.axis("off")